In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
FR_df = pd.read_csv(Path('../../data/main/master/product/.keep/slim_weekly_FR_before_dropping.csv')).sort_values(['week_id', 'nikkei']).reset_index(drop=True)

## ファンダメンタルズを算出

### 前準備として，決算データを整理する

In [ ]:
def group_func(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values('week_id').reset_index(drop=True)
    # 線形補完を使って欠損値を補完
    group.loc[group['親会社株主に帰属する当期純利益（連結）／当期利益（単独）'] == 0, '親会社株主に帰属する当期純利益（連結）／当期利益（単独）'] = np.nan
    group['親会社株主に帰属する当期純利益（連結）／当期利益（単独）'] = group['親会社株主に帰属する当期純利益（連結）／当期利益（単独）'].interpolate(method='linear', limit_direction='both')
    group['当期純利益'] = group['親会社株主に帰属する当期純利益（連結）／当期利益（単独）'] + group['非支配株主に帰属する当期純利益'].fillna(group['非支配株主に帰属する当期純利益'].ffill().bfill())
    group.rename(columns={'親会社株主に帰属する当期純利益（連結）／当期利益（単独）': '親会社株主に帰属する当期純利益'}, inplace=True)
    group['自己資本／親会社の所有者に帰属する資本'] = group['自己資本／親会社の所有者に帰属する資本'].interpolate(method='linear', limit_direction='both')
    group.rename(columns={'自己資本／親会社の所有者に帰属する資本': '自己資本'}, inplace=True)
    group['資産合計'] = group['資産合計'].interpolate(method='linear', limit_direction='both')
    group['資産合計'] = group['資産合計'].interpolate(method='linear', limit_direction='both')
    group['負債合計'] = group['負債合計'].interpolate(method='linear', limit_direction='both')
    group.loc[group['純資産合計／資本合計'].isna(), '純資産合計／資本合計'] = (
        group.loc[group['純資産合計／資本合計'].isna(), '資産合計'] -
        group.loc[group['純資産合計／資本合計'].isna(), '負債合計']
    )
    group.rename(columns={'純資産合計／資本合計': '純資産合計'}, inplace=True)
    group['減価償却費'] = group['減価償却費'].interpolate(method='linear', limit_direction='both')
    group['営業利益'] = group['営業利益'].interpolate(method='linear', limit_direction='both')
    group['有利子負債'] = group['有利子負債'].interpolate(method='linear', limit_direction='both')
    group['現金及び預金'] = group['現金及び預金'].interpolate(method='linear', limit_direction='both')
    group['純有利子負債'] = group['有利子負債'] - group['現金及び預金']
    group.iloc[group['上場フラグ'] == 0, 2:] = np.nan
    group[group['上場フラグ'] == np.nan] = 0
    return group

In [ ]:
FR_df = FR_df.groupby('nikkei').apply(group_func).reset_index(drop=True)

In [3]:
stcinfo_dir_path = Path('../../data/main/expansion/fundamental/etc/weekly_etc')
tr_csv = pd.read_csv(Path('../../data/main/master/product/weekly_TR(wed)_Tech.csv'))
invalid_mask = ~tr_csv.iloc[:, 2:].notnull().all(axis=1)

all_finrep_df = pd.DataFrame()
dst_dir_path = Path('../../data/main/master/product/.keep/weekly_stcinfo_V2.csv')
freq_id_name = 'week'
id_start_idx = 4

for i, fp in enumerate(sorted([fp for fp in stcinfo_dir_path.rglob('.') if fp.suffix == '.csv'], key=lambda x: int(x.stem[id_start_idx:]))):
    stcinfo_df = pd.read_csv(fp).drop(columns=['ticker', 'jp'])
    print('\r', f'Merging {freq_id_name} ID {fp.stem[id_start_idx:]} ({i+1}/1199) ... ', end='     ')
    stcinfo_df = pd.read_csv(fp)
    stcinfo_df[f'{freq_id_name}_id'] = fp.stem[id_start_idx:]
    all_finrep_df = pd.concat([all_finrep_df, stcinfo_df], ignore_index=True, axis=0)

print('\nAll fundamental csv merged.')
all_finrep_df = all_finrep_df[['nikkei', f'{freq_id_name}_id'] + [c for c in all_finrep_df.columns if c not in ['nikkei', f'{freq_id_name}_id', 'ticker', 'jp']]]
all_finrep_df = all_finrep_df.drop_duplicates(subset=['nikkei', f'{freq_id_name}_id'], keep='last').reset_index(drop=True) # 条件は決算データと同じ
all_finrep_df.iloc[invalid_mask, 2:] = np.nan
print('Invalid rows set to NaN.')

output_fp = Path(dst_dir_path)
all_finrep_df.to_csv(output_fp, index=False)

 Merging week ID 1198 (1199/1199) ...      
All fundamental csv merged.
Invalid rows set to NaN.


In [4]:
all_finrep_df[all_finrep_df['nikkei'] == 'N0000088'].count()

nikkei           1199
week_id          1199
期中平均株式数［累計］        68
１株当たり配当金（各期末）       0
dtype: int64

In [14]:
all_finrep_df.loc[(all_finrep_df['nikkei'] == 'N0000088') & (all_finrep_df['期中平均株式数［累計］'].notna()), ['期中平均株式数［累計］', '１株当たり配当金（各期末）']]

,期中平均株式数［累計］,１株当たり配当金（各期末）
184373,10000000.0,NaN
188981,10000000.0,NaN
193589,10000000.0,NaN
198197,10000000.0,NaN
202805,10000000.0,NaN
...,...,...
474677,9998691.0,NaN
479285,9998691.0,NaN
483893,9998691.0,NaN
488501,9998691.0,NaN


In [16]:
tr_csv[tr_csv['nikkei'] == 'N0000088'].count()

nikkei         1199
week_id        1199
open              0
weekly_high       0
weekly_low        0
close             0
SMA_5             0
SMA_15            0
SMA_40            0
RSI_26            0
RSI_14            0
RSI_9             0
MACD              0
MACD_signal       0
MACD_hist         0
STC_fk            0
STC_fd            0
MOM_10            0
BB2U              0
BB1U              0
BB1L              0
BB2L              0
dtype: int64

## 各書類のデータを統合

In [ ]:
fp = Path('../../data/main/master/product/.keep/weekly_stcinfo_v2.csv')
stcinfo_df = pd.read_csv(fp)
stcinfo_df

In [ ]:
TR_df = pd.read_csv(Path('../../data/main/master/product/weekly_TR(wed)_Tech.csv')).sort_values(['week_id', 'nikkei']).reset_index(drop=True)
TR_df[-20:]

In [ ]:
FR_df = FR_df.sort_values(by=['week_id', 'nikkei']).reset_index(drop=True)
FR_df[-20:]

In [ ]:
stcinfo_df.rename(columns={
    '期中平均株式数［累計］': '発行済株式数',
    '１株当たり配当金（各期末）': '１株当たり配当金'
}, inplace=True)

In [ ]:
# stcinfo_df['発行済株式数'] = stcinfo_df['発行済株式数'] # + FR_df['自己株式（▲）']
stcinfo_df['時価総額'] = TR_df['close'] * stcinfo_df['発行済株式数']
stcinfo_df['PER'] = TR_df['close'] / (FR_df['親会社株主に帰属する当期純利益']*1e06 / stcinfo_df['発行済株式数'])
stcinfo_df['EPS'] = FR_df['親会社株主に帰属する当期純利益']*1e06 / stcinfo_df['発行済株式数']
stcinfo_df['PBR'] = TR_df['close'] / (FR_df['純資産合計']*1e06 / stcinfo_df['発行済株式数'])
stcinfo_df['BPS'] = FR_df['純資産合計']*1e06 / stcinfo_df['発行済株式数']
stcinfo_df['PSR'] = stcinfo_df['時価総額'] / (FR_df['売上高・営業収益']*1e06)
stcinfo_df['SPS'] = (FR_df['売上高・営業収益']*1e06) / stcinfo_df['発行済株式数']
stcinfo_df['EV'] = stcinfo_df['時価総額'] + FR_df['純有利子負債']*1e06
stcinfo_df['CFPS'] = (FR_df['親会社株主に帰属する当期純利益']*1e06 + FR_df['減価償却費']*1e06) / stcinfo_df['発行済株式数']
stcinfo_df['PCFR'] = TR_df['close'] / stcinfo_df['CFPS']
stcinfo_df['EBITDA'] = FR_df['EBITDA']
stcinfo_df['EV/EBITDA'] = stcinfo_df['EV'] / stcinfo_df['EBITDA']
stcinfo_df['内部留保'] = (FR_df['親会社株主に帰属する当期純利益']*1e06 - stcinfo_df['１株当たり配当金']*stcinfo_df['発行済株式数']) / 1e06
stcinfo_df['サスティナブル成長率'] = FR_df['ROE'] * (stcinfo_df['内部留保'] / (FR_df['親会社株主に帰属する当期純利益']))
stcinfo_df['１株当たり配当金'] = stcinfo_df['１株当たり配当金']
stcinfo_df['配当性向'] = (stcinfo_df['１株当たり配当金'] * stcinfo_df['発行済株式数']) / (FR_df['親会社株主に帰属する当期純利益']*1e06)
stcinfo_df['ROE'] = FR_df['ROE']
stcinfo_df['ROA'] = FR_df['ROA']
stcinfo_df['自己資本比率'] = FR_df['自己資本比率']
stcinfo_df['流動比率'] = FR_df['流動比率']
stcinfo_df['固定比率'] = FR_df['固定比率']
stcinfo_df['総資産回転率'] = FR_df['総資産回転率']
stcinfo_df['株主資本回転率'] = FR_df['株主資本回転率']
stcinfo_df['ギアリング比率'] = FR_df['ギアリング比率']
stcinfo_df['フリーキャッシュフロー'] = FR_df['フリーキャッシュフロー']
stcinfo_df['売上高・営業収益の増加額'] = FR_df['売上高・営業収益'] - FR_df['売上高・営業収益'].shift(1)
stcinfo_df['売上総利益の増加額'] = FR_df['売上総利益']- FR_df['売上高・営業収益'].shift(1)
stcinfo_df['営業利益の増加額'] = FR_df['営業利益'] - FR_df['営業利益'].shift(1)
stcinfo_df['経常利益／税金等調整前当期純利益の増加額'] = FR_df['経常利益／税金等調整前当期純利益'] - FR_df['経常利益／税金等調整前当期純利益'].shift(1)
stcinfo_df['売上高総利益率'] = FR_df['売上高総利益率']
stcinfo_df['売上高営業利益率'] = FR_df['売上高営業利益率']
stcinfo_df['売上高経常利益率'] = FR_df['売上高経常利益率']
stcinfo_df['流動資産率'] = FR_df['流動資産率']
stcinfo_df['固定資産率'] = FR_df['固定資産率']
stcinfo_df['現金及び現金同等物の増加額'] = FR_df['現金及び現金同等物の期末残高'] - FR_df['現金及び現金同等物の期末残高'].shift(1)
stcinfo_df['純有利子負債'] = FR_df['純有利子負債']
stcinfo_df['株主資本回転率'] = FR_df['株主資本回転率']

In [ ]:
stcinfo_df.loc[stcinfo_df['PER'] > 10000, ['nikkei', 'week_id', 'PER', 'EPS']]

In [ ]:
stcinfo_df.loc[stcinfo_df['PER'].notnull(), 'PER'].describe()

In [ ]:
stcinfo_df.loc[stcinfo_df['PER'].notnull(), 'PER'].value_counts(bins=[10, 20, 100, 1000, 10000, 50000, np.inf]).sort_index()

In [ ]:
stcinfo_df.loc[:, 'PER'].max()

In [ ]:
stcinfo_df.loc[stcinfo_df['PER'] == np.inf, 'nikkei']

In [ ]:
print(len(list(set(stcinfo_df.loc[stcinfo_df['PER'] == np.inf, 'nikkei'].to_list()))))
sorted(list(set(stcinfo_df.loc[stcinfo_df['PER'] == np.inf, 'nikkei'].to_list())))

In [ ]:
delete_nikkei = ['N0000244', 'N0031178']

In [ ]:
# cutted_1y_Funda_df = stcinfo_df[stcinfo_df['week_id'] >= 40]
cutted_1y_Funda_df = stcinfo_df

In [ ]:
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['PER'] < 0, 'PER'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['EPS'] < 0, 'EPS'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['PBR'] < 0, 'PBR'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['BPS'] < 0, 'BPS'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['ROE'] < 0, 'ROE'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['ROA'] < 0, 'ROA'] = 0
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['配当性向'] < 0, '配当性向'] = 0

In [ ]:
# cutted_1y_Funda_df = cutted_1y_Funda_df[cutted_1y_Funda_df['nikkei'].isin(delete_nikkei) == False]

In [ ]:
cutted_1y_Funda_df.loc[cutted_1y_Funda_df['配当性向'].notnull(), '配当性向'].value_counts(bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0, 2.0, 3.0, np.inf]).sort_index()

In [ ]:
print(len(list(set(cutted_1y_Funda_df.loc[cutted_1y_Funda_df['PER'] == np.inf, 'nikkei'].to_list()))))
sorted(list(set(cutted_1y_Funda_df.loc[cutted_1y_Funda_df['PER'] == np.inf, 'nikkei'].to_list())))

In [ ]:
cutted_1y_Funda_df.to_csv(Path('../../data/main/master/product/weekly_FA_v2.csv'), index=False)

In [ ]:
drop_list = [
    '流動資産',
    '流動資産',
    '現金・預金／現金及び現金同等物',
    '受取手形・売掛金及び契約資産／売掛金及びその他の短期債権',
    'その他流動資産／その他の金融資産',
    '固定資産／非流動資産',
    '有形固定資産',
    '償却対象有形固定資産',
    '機械装置及び運搬具',
    '土地・その他非償却対象有形固定資産',
    '無形固定資産／無形資産',
    '投資・その他の資産合計',
    '投資有価証券・関係会社株式・出資金',
    '資産合計',
    '流動負債',
    '支払手形・買掛金／買掛金及びその他の短期債務',
    '短期借入金・社債合計',
    '１年内返済の借入金',
    '未払金・未払費用',
    'その他流動負債／その他の金融負債',
    '固定負債／非流動負債',
    '長期借入金・社債・転換社債',
    '社債・転換社債',
    '長期借入金',
    '長期未払金',
    '引当金合計',
    '退職給付に係る負債（退職給付引当金）',
    '再評価に係る繰延税金負債',
    '資産除去債務／資産除去債務引当金',
    '負債合計',
    '株主資本',
    '資本準備金',
    'その他利益剰余金',
    '任意積立金',
    '繰越利益剰余金',
    '自己株式（▲）',
    '評価・換算差額等／累積その他の包括利益',
    'その他有価証券再評価差額金／金融資産の公正価値',
    '土地再評価差額金',
    '為替換算調整勘定／在外営業活動体の換算差額',
    '非支配株主持分／非支配持分',
    '負債・純資産合計／資本及び負債合計',
    '売上高・営業収益',
    '売上原価・営業原価',
    '営業利益',
    '営業外収益',
    '有価証券売却益',
    '営業外費用',
    '経常利益／税金等調整前当期純利益',
    '特別利益',
    '減価償却費',
    '税金等調整前当期利益',
    '法人税等',
    '受取利息及び受取配当金（▲）',
    '支払利息',
    '為替差損（▲益）',
    '小計',
    '利息の支払額（▲）',
    '営業活動によるキャッシュフロー',
    '日経調整営業キャッシュフロー',
    '固定資産の取得による支出（▲）',
    '固定資産の売却による収入',
    '有価証券の売却による収入',
    '貸付金の回収による収入',
    '投資活動によるキャッシュフロー',
    '日経調整投資キャッシュフロー',
    '長期借入金の返済による支出（▲）',
    '財務活動によるキャッシュフロー',
    '日経調整財務キャッシュフロー',
    '現金及び現金同等物の増加額（▲減）',
    '現金及び現金同等物の期首残高',
    '現金及び現金同等物の期末残高',
    '現金及び預金',
    '親会社株主に帰属する当期純利益',

    '自己資本比率',
    '流動比率',
    '固定比率',
    '総資産回転率',
    'ギアリング比率',
    'フリーキャッシュフロー',
    '流動資産率',
    '固定資産率',
    '売上高総利益率',
    '売上高営業利益率',
    '売上高経常利益率',
    '有利子負債',
    '株主資本回転率',
    'ROE',
    'ROA',
    'EBITDA',  
    '純有利子負債',
]

In [ ]:
FR_df.drop(columns=drop_list, inplace=True)

In [ ]:
print(FR_df.columns.tolist()[2:])

In [ ]:
len(FR_df.columns.tolist()[2:])

In [ ]:
len(FR_df)

In [ ]:
FR_df.to_csv(Path('../../data/main/master/product/slim_weekly_FR_v2.csv'), index=False)